In [ ]:
import os
from datetime import datetime
import pickle
import random
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MaxAbsScaler, MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F



In [ ]:
df = pd.read_csv(f"BTCUSDT-15m-data.csv")
print(df)

**Check missing values**

In [ ]:
print(df.isnull())
print(f"Counts how many missing values there are in each column: {df.isnull().sum()}")
print(f"Total missing values: {df.isnull().sum().sum()}")

**Split data**

In [ ]:
train_end_idx = 240_000

df_train = df.iloc[:train_end_idx].copy()
df_test = df.iloc[train_end_idx:].copy()

df_test.reset_index(drop=True, inplace=True)

print(f"Length of df_train: {len(df_train)}")
print(f"Length of df_test: {len(df_test)}")

**Building Meaningful Features**

In [ ]:
def build_features(opens, closes):
    feature1 = (closes - opens) / opens

    # Stack features
    features = np.stack([
        feature1,
        # ...
    ], axis=-1) # shape: (time_steps, num_features)

    num_features = features.shape[-1]
    return features, num_features

**Preprocess data**

In [ ]:
def preprocess_data(seq_len, df):
    m = len(df)

    opens = np.array(df['open'].values)
    closes = np.array(df['close'].values)

    # Build features
    features, num_features = build_features(opens, closes)

    # Calculate number of samples
    num_samples = m - seq_len

    # Create storage for inputs & targets
    X = np.zeros([num_samples, seq_len, num_features], dtype=np.float32)
    Y = np.zeros([num_samples, num_features], dtype=np.float32)

    # Create samples (X, Y)
    for i in range(num_samples):
        X[i] = features[i : i+seq_len]
        Y[i] = features[i+seq_len : i+seq_len+1]

    return X, Y, num_features

In [ ]:
# Sequence Length
seq_len = 96

# Preprocess data
X_train, Y_train, num_features = preprocess_data(seq_len, df_train)
X_test, Y_test, num_features = preprocess_data(seq_len, df_test)


In [ ]:
m_train = X_train.shape[0]
m_test = X_test.shape[0]
print(f"m_train: {m_train}")
print(f"m_test: {m_test}")
print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

**Transform data to Torch Tensor**

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else 'cpu')
print("my deive: ", device)

X_train = torch.from_numpy(X_train.astype(np.float32)).to(device, dtype=torch.float32)
Y_train = torch.from_numpy(Y_train.astype(np.float32)).to(device, dtype=torch.float32)

X_test = torch.from_numpy(X_test.astype(np.float32)).to(device, dtype=torch.float32)
Y_test = torch.from_numpy(Y_test.astype(np.float32)).to(device, dtype=torch.float32)

Y_pred_test = torch.zeros([m_test, num_features], device=device, dtype=torch.float32)